In [1]:
import pandas as pd
import numpy as np
from jax import numpy as jnp
from jax.random import PRNGKey

pd.options.plotting.backend = "plotly"

from summer3.graph import CompartmentValues, defer, Parameter
from summer3.epi import TransitionFlow, Stratification, CompartmentMap, CompartmentalEpiModel, CategoryData, CompartmentalModelODE, build_istate, dti_to_epoch

In [2]:
start_time = 0.0
end_time = 70.0
model_comps = [
    "vaccinated", 
    "susceptible", 
    "infectious_asympt", 
    "infectious_sympt", 
    "recovered",
]
infect_comps = [
    "infectious_asympt", 
    "infectious_sympt",
]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)

In [3]:
def infection_process(compartment_values, disease_state):
    total_pop = compartment_values.sum()
    # i_pop = compartment_values.sumcats(disease_state["infectious_asympt"])
    # foi = i_pop / total_pop
    return 1.0

foi = defer(infection_process)(CompartmentValues, disease_state)

infection = TransitionFlow(
    "infection", 
    disease_state["susceptible"], 
    disease_state["infectious_asympt"], 
    foi,
)
recovery = TransitionFlow(
    "recovery",
    disease_state["infectious_asympt"],
    disease_state["recovered"],
    1.0 / Parameter("recovery_time", 10.0),
)

times = pd.date_range("7 jun 1980", "7 december 1980")
epi_model = CompartmentalEpiModel(humans, times)
init_pops = CategoryData(disease_state.categories(), jnp.array(([0.9, 0.1, 0.0, 0.0, 0.0])))
epi_model.set_initial_population(init_pops)
epi_model.add_flow(infection)
epi_model.add_flow(recovery)

In [4]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

params = {"contact_rate": 0.2, "recovery_time": 20.0}
runner, istate = get_runner(epi_model)
results = epi_model.run(params)

In [5]:
def get_derived_results(params):
    results = epi_model.run(params)  # runner.run(istate.data, params)
    inf_flow = results["flows"]["infection"]
    weekly_target = (
        inf_flow.sum(to_dims="time").rolling(7, jnp.sum).query(time=inf_target.index)
    )
    return weekly_target